# Multi-class image classification: reviewed and optimized notebook

This version keeps `ReLU` as the activation function throughout the network and is tuned for an `89%+` validation-accuracy target on the challenge data.

The main changes are:
- robust data loading that works in Colab and outside Colab,
- a stratified validation split so every class is represented fairly,
- normalization from the training split instead of fixed `0.5 / 0.5`,
- conservative small-image augmentation that preserves class identity,
- a ReLU residual CNN designed for `16x16` inputs,
- a stronger optimization loop with `AdamW`, `OneCycleLR`, mixed precision, gradient clipping, and best-checkpoint restore.


In [ ]:
import copy
import os
import pickle
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

# Reproducibility matters when tuning a model toward a target score.
# Fixing seeds makes comparisons between experiments much more reliable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Load data and make the notebook portable

Why this step matters:
- The original notebook depended directly on `google.colab`, which breaks outside Colab.
- A portable path resolver makes the notebook easier to rerun while keeping the same training code.
- We also validate that both `.pkl` files exist before training starts, so failures happen early and clearly.


In [ ]:
def resolve_base_path():
    """Locate the directory that contains train_X_y.pkl and test_X.pkl."""
    candidates = []

    env_path = os.environ.get("CHALLENGE_DATA_DIR")
    if env_path:
        candidates.append(Path(env_path).expanduser())

    cwd = Path.cwd()
    candidates.extend([
        cwd,
        cwd / "data",
        cwd / "dataset",
        cwd / "W2026-0-0.3",
        Path("/content/drive/MyDrive/image-classification-challenge-w-26/W2026-0-0.3"),
    ])

    def has_required_files(path):
        return (path / "train_X_y.pkl").exists() and (path / "test_X.pkl").exists()

    for candidate in candidates:
        if has_required_files(candidate):
            return candidate

    # If the notebook is running in Colab, try mounting Drive as a fallback.
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
        drive_candidate = Path("/content/drive/MyDrive/image-classification-challenge-w-26/W2026-0-0.3")
        if has_required_files(drive_candidate):
            return drive_candidate
    except ImportError:
        pass

    checked_locations = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        "Could not find train_X_y.pkl and test_X.pkl. Checked:\n"
        f"{checked_locations}\n\n"
        "Tip: put the files in one of the locations above or set CHALLENGE_DATA_DIR."
    )


base_path = resolve_base_path()
print(f"Data directory: {base_path}")


In [ ]:
# Training data
with open(base_path / 'train_X_y.pkl', 'rb') as f:
    X_train, y_train = pickle.load(f)

# Test data
with open(base_path / 'test_X.pkl', 'rb') as f:
    X_test = pickle.load(f)

print("Raw X_train shape:", X_train.shape)
print("Raw y_train shape:", y_train.shape)
print("Raw X_test shape :", X_test.shape)


In [ ]:
def to_nchw(array, name):
    """Convert images to NCHW because PyTorch conv layers expect channels first."""
    if array.ndim != 4:
        raise ValueError(f"{name} must be a 4D array, got shape {array.shape}")

    if array.shape[1] == 3:
        converted = array.astype(np.float32)
    elif array.shape[-1] == 3:
        converted = np.transpose(array, (0, 3, 1, 2)).astype(np.float32)
    else:
        raise ValueError(
            f"{name} must have 3 channels in either axis 1 or axis -1, got shape {array.shape}"
        )

    return converted


X_train = to_nchw(X_train, "X_train")
X_test = to_nchw(X_test, "X_test")
y_train = y_train.reshape(-1).astype(np.int64)

IMAGE_SIZE = X_train.shape[-1]
NUM_CLASSES = len(np.unique(y_train))

print("X_train shape after conversion:", X_train.shape)
print("X_test shape after conversion :", X_test.shape)
print("y_train shape after flatten   :", y_train.shape)
print("Pixel value range             :", float(X_train.min()), "to", float(X_train.max()))
print("Number of classes             :", NUM_CLASSES)


In [ ]:
# Small images need careful augmentation.
# We avoid aggressive transforms such as vertical flips or heavy rotations
# because they can change the class semantics at 16x16 resolution.
def reflect_pad_crop(x, padding):
    if padding <= 0:
        return x
    x = F.pad(x, (padding, padding, padding, padding), mode="reflect")
    top = int(torch.randint(0, 2 * padding + 1, (1,)).item())
    left = int(torch.randint(0, 2 * padding + 1, (1,)).item())
    return x[:, top:top + IMAGE_SIZE, left:left + IMAGE_SIZE]


def random_cutout(x, fill_value, p=0.25, min_size=2, max_size=4):
    if torch.rand(1).item() >= p:
        return x

    cutout_size = int(torch.randint(min_size, max_size + 1, (1,)).item())
    top = int(torch.randint(0, IMAGE_SIZE - cutout_size + 1, (1,)).item())
    left = int(torch.randint(0, IMAGE_SIZE - cutout_size + 1, (1,)).item())
    x[:, top:top + cutout_size, left:left + cutout_size] = fill_value.view(3, 1, 1)
    return x


class ImageDataset(Dataset):
    def __init__(self, images, labels=None, mean=None, std=None, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)
        self.cutout_fill = self.mean[:, 0, 0]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx]).float() / 255.0

        if self.augment:
            # Horizontal flips are usually safe for natural-image classes.
            if torch.rand(1).item() < 0.5:
                x = torch.flip(x, dims=[2])

            # Reflection padding + random crop acts like small translations
            # without destroying the already tiny 16x16 content.
            x = reflect_pad_crop(x, padding=2)

            # Mild Gaussian noise improves robustness but stays subtle.
            if torch.rand(1).item() < 0.20:
                noise = torch.randn_like(x) * 0.02
                x = torch.clamp(x + noise, 0.0, 1.0)

            # Small cutout encourages the network to use more than one cue.
            x = random_cutout(x, fill_value=self.cutout_fill, p=0.25)

        x = (x - self.mean) / self.std

        if self.labels is None:
            return x, torch.tensor(-1, dtype=torch.long)

        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


In [ ]:
# A stratified split keeps class proportions stable in both training and
# validation. That makes validation accuracy more trustworthy.
VAL_SIZE = 0.12
BATCH_SIZE = 96
NUM_WORKERS = min(2, os.cpu_count() or 0)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train,
)

# Use only the training split to estimate normalization statistics.
# This avoids leaking information from validation into preprocessing.
train_pixels = X_tr / 255.0
channel_mean = train_pixels.mean(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
channel_std = train_pixels.std(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
channel_std = np.clip(channel_std, 1e-6, None)

print("Channel mean:", channel_mean)
print("Channel std :", channel_std)

train_dataset = ImageDataset(X_tr, y_tr, mean=channel_mean, std=channel_std, augment=True)
val_dataset = ImageDataset(X_val, y_val, mean=channel_mean, std=channel_std, augment=False)
test_dataset = ImageDataset(X_test, labels=None, mean=channel_mean, std=channel_std, augment=False)

trainloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
valloader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
testloader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")
print(f"Test samples : {len(test_dataset)}")
print(f"Batch size   : {BATCH_SIZE}")


### Display a batch after preprocessing

Why this step matters:
- It confirms that augmentation is active for the training loader.
- It helps catch common bugs such as broken normalization or incorrect channel ordering.
- Looking at a few examples is a cheap sanity check before a long training run.


In [ ]:
def imshow(img):
    """Undo normalization so the grid looks like an image again."""
    mean = torch.tensor(channel_mean).view(3, 1, 1)
    std = torch.tensor(channel_std).view(3, 1, 1)
    img = img * std + mean
    img = torch.clamp(img, 0.0, 1.0)
    npimg = img.numpy()
    plt.figure(figsize=(12, 3))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')
    plt.title("Sample training images after augmentation + normalization")
    plt.show()


dataiter = iter(trainloader)
images, labels = next(dataiter)
imshow(torchvision.utils.make_grid(images[:16]))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


## 2. Define a ReLU CNN that is better matched to tiny images

Review of the original notebook:
- `RandomVerticalFlip`, strong rotation, and strong color jitter can be too destructive on `16x16` inputs.
- Fixed normalization with `(0.5, 0.5, 0.5)` is workable, but train-split statistics usually converge faster.
- A residual network is a good choice here, but it benefits more when paired with a stronger optimizer schedule.
- The new version keeps `ReLU` exactly as requested and focuses on stable optimization rather than changing the activation family.

Model design rationale:
- We keep high spatial resolution early, because `16x16` images lose information quickly if pooled too soon.
- Residual blocks make optimization easier and let us use a deeper model without unstable gradients.
- Channel widths grow from `64 -> 128 -> 256` so the network can learn richer features as the image gets smaller.
- Global average pooling reduces overfitting compared with a large fully connected head.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.dropout(out)
        out = self.relu(out + identity)
        return out


class SmallImageResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        # Two stem convolutions let the model extract local edges and color
        # patterns before any downsampling happens.
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Stage 1 keeps the full 16x16 resolution so we do not throw away
        # fine detail too early.
        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64, stride=1, dropout=0.05),
            ResidualBlock(64, 64, stride=1, dropout=0.05),
            nn.MaxPool2d(2),
        )

        # Stage 2 and Stage 3 reduce spatial size gradually while increasing
        # channel capacity. This is a standard and effective small-image pattern.
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2, dropout=0.10),
            ResidualBlock(128, 128, stride=1, dropout=0.10),
        )
        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2, dropout=0.15),
            ResidualBlock(256, 256, stride=1, dropout=0.15),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes),
        )

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


net = SmallImageResNet(num_classes=NUM_CLASSES)
print(net)


In [ ]:
# Move the model once so every later cell uses the same device placement.
net = net.to(device)
print(f"Model moved to: {device}")


In [ ]:
# Optional model summary. If torchinfo is not installed, fall back to a
# manual parameter count so the notebook still runs cleanly.
try:
    from torchinfo import summary

    summary(net, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE))
except ImportError:
    total_params = sum(p.numel() for p in net.parameters())
    trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"Total parameters    : {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")


## 3. Define the loss, optimizer, and learning-rate schedule

Why these choices:
- `CrossEntropyLoss` with light label smoothing reduces over-confidence without changing the task.
- `AdamW` is a strong default for image models because it handles adaptive steps well and separates weight decay from the gradient update.
- `OneCycleLR` changes the learning rate every mini-batch instead of every epoch, which is usually more effective for reaching a strong score in fewer epochs.
- Gradient clipping protects training from occasional unstable updates when the schedule is near its peak learning rate.


In [ ]:
TARGET_VAL_ACCURACY = 89.0
EPOCHS = 40
MAX_LR = 1e-3
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05
EARLY_STOPPING_PATIENCE = 10
MAX_GRAD_NORM = 1.0

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(net.parameters(), lr=MAX_LR / 10, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    epochs=EPOCHS,
    steps_per_epoch=len(trainloader),
    pct_start=0.20,
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=100.0,
)
scaler = GradScaler(enabled=torch.cuda.is_available())

print(f"Loss                    : CrossEntropyLoss(label_smoothing={LABEL_SMOOTHING})")
print(f"Optimizer               : AdamW(lr={MAX_LR / 10}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler               : OneCycleLR(max_lr={MAX_LR}, epochs={EPOCHS})")
print(f"Target validation score : {TARGET_VAL_ACCURACY:.1f}%")


## 4. Train with validation, checkpointing, and early stopping

Why this training loop is stronger than the original one:
- the scheduler updates every batch rather than every epoch,
- the best validation checkpoint is restored before final evaluation,
- mixed precision speeds up GPU training without changing the model,
- early stopping saves time if the score plateaus,
- we track both loss and accuracy so it is easier to diagnose underfitting or overfitting.


In [ ]:
def run_train_epoch(model, loader):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=scaler.is_enabled()):
            logits = model(inputs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * inputs.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc


def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(enabled=scaler.is_enabled()):
                logits = model(inputs)
                loss = criterion(logits, labels)

            preds = torch.argmax(logits, dim=1)
            running_loss += loss.item() * inputs.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc, np.array(all_preds), np.array(all_labels)


best_val_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
best_state = None

train_loss_hist = []
train_acc_hist = []
val_loss_hist = []
val_acc_hist = []
lr_hist = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_train_epoch(net, trainloader)
    val_loss, val_acc, _, _ = evaluate(net, valloader)
    current_lr = optimizer.param_groups[0]["lr"]

    train_loss_hist.append(train_loss)
    train_acc_hist.append(train_acc)
    val_loss_hist.append(val_loss)
    val_acc_hist.append(val_acc)
    lr_hist.append(current_lr)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}% | "
        f"LR: {current_lr:.2e}"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping triggered after {epoch} epochs "
            f"without validation improvement."
        )
        break

if best_state is not None:
    net.load_state_dict(best_state)
    net.to(device)

print(f"\nBest validation accuracy: {best_val_acc:.2f}% at epoch {best_epoch}")
if best_val_acc >= TARGET_VAL_ACCURACY:
    print("Target reached: validation accuracy is at or above 89%.")
else:
    print(
        "Target not reached in this run. The code is tuned to push toward 89%+, "
        "but the final score still depends on the actual dataset, runtime, and hardware."
    )


In [ ]:
# These plots help explain how the training recipe behaved.
# In particular, compare train/validation curves to spot overfitting.
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
epochs_ran = range(1, len(train_loss_hist) + 1)

axes[0, 0].plot(epochs_ran, train_loss_hist, color='steelblue', label='Train loss')
axes[0, 0].plot(epochs_ran, val_loss_hist, color='darkorange', label='Val loss')
axes[0, 0].set_title('Loss by epoch')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Cross-entropy loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_ran, train_acc_hist, color='seagreen', label='Train acc')
axes[0, 1].plot(epochs_ran, val_acc_hist, color='crimson', label='Val acc')
axes[0, 1].axhline(y=TARGET_VAL_ACCURACY, color='black', linestyle='--', label='89% target')
axes[0, 1].set_title('Accuracy by epoch')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_ran, lr_hist, color='purple')
axes[1, 0].set_title('Learning-rate schedule')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning rate')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

generalization_gap = np.array(train_acc_hist) - np.array(val_acc_hist)
axes[1, 1].plot(epochs_ran, generalization_gap, color='teal')
axes[1, 1].axhline(y=0.0, color='black', linestyle='--')
axes[1, 1].set_title('Train - val accuracy gap')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Gap (percentage points)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Evaluate on the held-out validation set

Why this step matters:
- The best checkpoint is chosen using validation accuracy, so we re-evaluate after restoring it.
- Per-class accuracy helps explain where the model is still weak even if the overall score is strong.
- This is the number to compare against the `89%` target during development.


In [ ]:
val_loss, overall_acc, all_preds, all_labels = evaluate(net, valloader)
print(f"Validation loss: {val_loss:.4f}")
print(f"Overall validation accuracy: {overall_acc:.2f}%")

print("\nPer-class accuracy:")
for cls in range(NUM_CLASSES):
    mask = all_labels == cls
    cls_count = int(mask.sum())
    cls_acc = 100.0 * np.mean(all_preds[mask] == all_labels[mask]) if cls_count > 0 else 0.0
    print(f"  Class {cls}: {cls_acc:6.2f}% ({cls_count} samples)")


## 6. Evaluate on the full training set without augmentation

Why this step matters:
- A much higher training score than validation score usually means overfitting.
- Evaluating without augmentation gives a cleaner apples-to-apples comparison with validation.
- The train-vs-validation gap helps decide whether to add more regularization or more capacity.


In [ ]:
full_train_eval = ImageDataset(X_train, y_train, mean=channel_mean, std=channel_std, augment=False)
full_train_loader = DataLoader(
    full_train_eval,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

train_eval_loss, train_acc, _, _ = evaluate(net, full_train_loader)
print(f"Training loss without augmentation: {train_eval_loss:.4f}")
print(f"Training accuracy of the network : {train_acc:.2f}%")
print(f"Gap to validation accuracy       : {train_acc - overall_acc:.2f} percentage points")


## 7. Generate test predictions for submission

Why this step is last:
- We only want to export predictions after restoring the best validation checkpoint.
- That keeps the submission aligned with the most generalizable model seen during training.
- Reusing one prediction helper also keeps validation and test inference consistent.


In [ ]:
def predict_loader(model, loader):
    model.eval()
    output_labels = []

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device, non_blocking=True)
            with autocast(enabled=scaler.is_enabled()):
                logits = model(images)
            preds = torch.argmax(logits, dim=1)
            output_labels.extend(preds.cpu().numpy().tolist())

    return output_labels


output_labels = predict_loader(net, testloader)
print(f"Generated {len(output_labels)} test predictions.")
print(f"Unique predicted classes: {sorted(set(output_labels))}")


In [ ]:
test_predictions = pd.DataFrame({
    'rowId': np.arange(len(output_labels), dtype=np.int64),
    'label': np.array(output_labels, dtype=np.int64),
})

output_csv = base_path / 'my_cnn_results_improved.csv'
test_predictions.to_csv(output_csv, index=False)

print(test_predictions.head())
print(f"\nPredictions saved to: {output_csv}")


## 8. Save and reload the trained model

Why save only `state_dict()`:
- it is the standard PyTorch format,
- it keeps the file smaller,
- it avoids pickling the full Python object graph,
- it makes reloading explicit and reproducible.


In [ ]:
model_path = Path.cwd() / 'my_cnn_model_improved.pth'
torch.save(net.state_dict(), model_path)
print(f"Model saved to {model_path}")


In [ ]:
# To restore the model later, recreate the architecture first and then
# load the saved weights.
#
# reloaded_net = SmallImageResNet(num_classes=NUM_CLASSES)
# reloaded_net.load_state_dict(torch.load(model_path, map_location=device))
# reloaded_net.to(device)
# reloaded_net.eval()
# print("Model reloaded and ready for inference.")
